# Transfer and compress data for upload to zenodo

Including all data necessary to replicate analysis: 
- `aggs`
- ~`impact_projs`~ (too big, can be regerated from `aggs`)
- `reg_inputs`
- `reg_outputs` (only pop_wght, only loos on gdppc, for simplicty, without mortality robustness checks (`noFEs`, `wInd`, etc.)
- `uncert_parts`
- ~`aux`~ (regenerated temporary space, same with `tmp`)

In [27]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import re
import shutil
from tqdm.notebook import tqdm
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

In [2]:
from distributed import Client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /node/crhtc53.hpc.ucar.edu/40940/proxy/8787/status,
Dashboard: /node/crhtc53.hpc.ucar.edu/40940/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 144.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34991,Workers: 0
Dashboard: /node/crhtc53.hpc.ucar.edu/40940/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33189,Total threads: 3
Dashboard: /node/crhtc53.hpc.ucar.edu/40940/proxy/42931/status,Memory: 24.00 GiB
Nanny: tcp://127.0.0.1:43965,


In [96]:
dir_in = '/glade/derecho/scratch/schwarzwald/impact_uncertainty/'
dir_out = '/glade/work/schwarzwald/impact_uncertainty_upload/'
# With skiplists
transfer_dirs = {'aggs':{'keepre':r'popwght',
                         'skipre':r'\/tmp_safe\/'},
                 'uncert_parts':{},
                 'reg_inputs':{'skipre':(r'rawwght')},
                 'reg_outputs':{'skipre':(r'(\-fullsamp)|(mortality\/loos)|(payroll\/loos)|'+
                             r'(notFEs)|(noIND)|(\/tmp\/)|(old\_files)|(\_raw\_)')}}

In [97]:
dir_tmp = 'uncert_parts'


In [100]:
def copy_store(src, dst, workers=18):
    # Copy zarr store script
    src, dst = Path(src), Path(dst)
    files = [p for p in src.rglob("*") if p.is_file() and p.suffix != ".partial"]
    for d in {(dst / p.relative_to(src)).parent for p in files}:
        d.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(lambda p: shutil.copyfile(p, dst / p.relative_to(src)), files))

In [127]:
for dir_tmp in transfer_dirs:
    print(f'*** Processing directory {dir_tmp}...')

    # Get paths
    paths_tmp = []
    for root, paths0, files in os.walk(f'{dir_in}{dir_tmp}/'):
        zarr_dirs = [d for d in paths0 if d.endswith(".zarr")]
        
        # Zarr-Stores selbst aufnehmen
        paths_tmp.extend(os.path.join(root, d) for d in zarr_dirs)
        
        # Aber nicht in sie hineinlaufen
        paths0[:] = [d for d in paths0 if d not in zarr_dirs]
        
        paths_tmp.extend(os.path.join(root, f) for f in files)

    transfer_params = transfer_dirs[dir_tmp]
    if 'skipre' in transfer_params:
        skipre = transfer_params['skipre']
    else:
        skipre = None
    if 'keepre' in transfer_params:
        keepre = transfer_params['keepre']
    else:
        keepre = None
    #skipre = None
    #keepre = r'popwght'
    fns_keep = [fn for fn in paths_tmp if (((not re.search(skipre,fn)) if (skipre is not None) else True) and 
                                ((re.search(keepre,fn)) if (keepre is not None) else True))]

    storetypes = np.unique([re.split(r'\.',fn)[-1] for fn in fns_keep])
    nfns_bytype = {typ:len([fn for fn in fns_keep if fn.endswith(typ)])
                   for typ in storetypes}
    print(f'Copying the following # of stores/files:')
    for k in nfns_bytype:
        print(f'   {k}: {nfns_bytype[k]}')


    # Copy
    for fn_in in tqdm(fns_keep):
        fn_out = re.sub(dir_in,dir_out,fn_in)
        if not os.path.exists(Path(fn_out).parent):
            Path(fn_out).parent.mkdir(parents=True, exist_ok=True)
        if fn_in.endswith('.zarr'):
            copy_store(fn_in,fn_out)
        else:
            shutil.copyfile(fn_in,fn_out)

*** Processing directory aggs...
Copying the following # of stores/files:
   zarr: 270


  0%|          | 0/270 [00:00<?, ?it/s]

*** Processing directory uncert_parts...
Copying the following # of stores/files:
   zarr: 16


  0%|          | 0/16 [00:00<?, ?it/s]

*** Processing directory reg_inputs...
Copying the following # of stores/files:
   csv: 6


  0%|          | 0/6 [00:00<?, ?it/s]

*** Processing directory reg_outputs...
Copying the following # of stores/files:
   csv: 1986
   dta: 24


  0%|          | 0/2010 [00:00<?, ?it/s]

In [128]:
dirlist_out = [f'{dir_out}{dir_tmp}' for dir_tmp in transfer_dirs]

In [130]:
for dir_out_tmp in dirlist_out:
    print(f'Compressing {dir_out_tmp}...')
    os.system(f'tar -cf - {dir_out_tmp} | zstd -T0 -1 -o {dir_out_tmp}.tar.zst')

Compressing /glade/work/schwarzwald/impact_uncertainty_upload/aggs...


tar: Removing leading `/' from member names
/*stdin*\            : 99.49%   (  16.6 GiB =>   16.5 GiB, /glade/work/schwarzwald/impact_uncertainty_upload/aggs.tar.zst) 
tar: Removing leading `/' from member names


Compressing /glade/work/schwarzwald/impact_uncertainty_upload/uncert_parts...


/*stdin*\            : 99.82%   (  4.16 GiB =>   4.16 GiB, /glade/work/schwarzwald/impact_uncertainty_upload/uncert_parts.tar.zst) 
tar: Removing leading `/' from member names


Compressing /glade/work/schwarzwald/impact_uncertainty_upload/reg_inputs...


/*stdin*\            : 16.82%   (  1.60 GiB =>    275 MiB, /glade/work/schwarzwald/impact_uncertainty_upload/reg_inputs.tar.zst) 
tar: Removing leading `/' from member names


Compressing /glade/work/schwarzwald/impact_uncertainty_upload/reg_outputs...


/*stdin*\            :  8.24%   (  4.68 MiB =>    395 KiB, /glade/work/schwarzwald/impact_uncertainty_upload/reg_outputs.tar.zst) 
